# Sprint 3: Analyzing the E-Commerce Dataset in SQL

### Project Mission
You are a **Database & Analytics Engineer**. In Sprint 2, we cleaned our product listings. Now, we model this data into a production-grade relational database (SQLite via SQLAlchemy), ingest customer review data, and solve business intelligence queries using advanced SQL: **Window Functions**, **Common Table Expressions (CTEs)**, and **In-Database Data Cleaning**.

---

### Relational Schema Architecture (ERD)

```text
+-------------------------+
|       categories        |
+-------------------------+
| PK  category_id         |<--------+
|     category            |         |
+-------------------------+         | 1 : N (One category has many products)
                                    |
+-------------------------+         |
|        products         |         |
+-------------------------+         |
| PK  product_id          |<---+    |
|     name                |    |    |
|     brand               |    |    |
|     price               |    |    |
|     rating              |    |    |
|     review_count        |    |    |
| FK  category_id         +----+----+
+-------------------------+    |
                               | 1 : N (One product has many reviews)
+-------------------------+    |
|         reviews         |    |
+-------------------------+    |
| PK  review_id           |    |
| FK  listing_id          +----+
|     review_date         |
|     rating              |
|     review_text         |
+-------------------------+
```


## Q1: How do we design a normalized 3-table schema to eliminate data redundancy?

### Concept Pointers:
- In Sprint 2's flat CSV, category names (e.g. `"electronics"`) were repeated on every row.
- Normalization splits repeated categories into a lookup table (`categories`) and references them via a foreign key (`category_id`).

## Q2: How do we load our Sprint 2 clean CSV and raw reviews into SQLite?

### Concept Pointers:
- Step-by-step database initialization:
  1. Read the CSV files with Pandas.
  2. Extract unique categories.
  3. Map `category_id` onto products.
  4. Write to SQLite using SQLAlchemy.

In [1]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine

DB_PATH = "ecommerce.db"
engine = create_engine(f"sqlite:///{DB_PATH}")

# Step 1: Read input datasets
products_raw = pd.read_csv("data/ecommerce_cleaned.csv")
reviews_raw = pd.read_csv("data/reviews_raw.csv")
print(f"Loaded {len(products_raw)} products and {len(reviews_raw)} reviews from CSV.")


Loaded 103 products and 595 reviews from CSV.


In [2]:
# Step 2: Build normalized categories table
categories = products_raw[["category"]].drop_duplicates().reset_index(drop=True)
categories["category_id"] = categories.index + 1
categories


,category,category_id
0,home-goods,1
1,electronics,2
2,sports-outdoors,3


In [3]:
# Step 3: Link category_id to products table
products = products_raw.merge(categories, on="category")[
    ["listing_id", "name", "brand", "price", "rating", "review_count", "category_id"]
].rename(columns={"listing_id": "product_id"})
products.head(3)


,product_id,name,brand,price,rating,review_count,category_id
0,1,Maple Row Electric Kettle,Maple Row,23.29,3.7,1269.0,1
1,2,Lumen Air Purifier,Lumen,186.52,2.9,2408.0,1
2,4,Maple Row Wall Clock,Maple Row,127.00,4.7,859.0,1


In [4]:
# Step 4: Write all 3 tables to SQLite database
categories.to_sql("categories", engine, if_exists="replace", index=False)
products.to_sql("products", engine, if_exists="replace", index=False)
reviews_raw.to_sql("reviews", engine, if_exists="replace", index=False)
print("[DATABASE] All 3 tables saved into ecommerce.db!")


[DATABASE] All 3 tables saved into ecommerce.db!


In [5]:
# Step 5: Query helper function
conn = sqlite3.connect(DB_PATH)

def q(sql_query):
    """Execute query and return formatted DataFrame."""
    return pd.read_sql_query(sql_query, conn)

q("SELECT COUNT(*) AS total_products FROM products")


,total_products
0,103


## Q3: How do we rank products within each category using SQL Window Functions?

### Concept Pointers:
- `GROUP BY` collapses rows.
- **Window Functions** (`RANK() OVER (PARTITION BY ... ORDER BY ...)`) compute rankings across groups while **preserving every individual product row**.

In [6]:
q("""
SELECT
    p.name,
    c.category,
    p.price,
    RANK() OVER (
        PARTITION BY c.category 
        ORDER BY p.price DESC
    ) AS price_rank_in_category
FROM products p
JOIN categories c ON p.category_id = c.category_id
ORDER BY c.category, price_rank_in_category
LIMIT 8;
""")


,name,category,price,price_rank_in_category
0,Vantage Wireless Mouse,electronics,2580.28,1
1,Nexlon Mechanical Keyboard,electronics,441.00,2
2,Orbis Smart Watch,electronics,432.94,3
3,Orbis 4K Monitor,electronics,428.35,4
4,Sonyx USB-C Hub,electronics,399.20,5
5,Brightek USB-C Hub,electronics,377.87,6
6,Sonyx Mechanical Keyboard,electronics,358.70,7
7,Brightek Bluetooth Speaker,electronics,318.21,8


## Q4: How do we write Common Table Expressions (CTEs) for a "Top Movers" report?

### Concept Pointers:
- **CTE (`WITH ... AS`)**: A temporary named query block that makes multi-step calculations easy to read.
- Step 1 computes category average price; Step 2 computes the gap between product price and category average.

In [7]:
q("""
WITH category_avg AS (
    SELECT category_id, AVG(price) AS avg_price
    FROM products
    GROUP BY category_id
),
price_gap_calc AS (
    SELECT 
        p.name,
        p.price,
        c.avg_price,
        ROUND(p.price - c.avg_price, 2) AS price_gap
    FROM products p
    JOIN category_avg c ON p.category_id = c.category_id
)
SELECT name, price, ROUND(avg_price, 2) AS category_avg, price_gap
FROM price_gap_calc
ORDER BY price_gap DESC
LIMIT 8;
""")


,name,price,category_avg,price_gap
0,Vantage Wireless Mouse,2580.28,286.30,2293.98
1,TrailBlaze Yoga Mat,379.00,188.09,190.91
2,Ironclad Running Shoes,370.76,188.09,182.67
3,TrailBlaze Adjustable Dumbbells,362.06,188.09,173.97
4,TrailBlaze Resistance Bands Set,359.00,188.09,170.91
5,Nexlon Mechanical Keyboard,441.00,286.30,154.70
6,Orbis Smart Watch,432.94,286.30,146.64
7,Orbis 4K Monitor,428.35,286.30,142.05


## Q5: How do we find duplicate reviews and safely delete them using `ROW_NUMBER() OVER`?

### Concept Pointers:
- If multiple rows are identical, a raw `DELETE` would remove all copies.
- We use `ROW_NUMBER() OVER (PARTITION BY listing_id, review_date, rating, review_text ORDER BY review_id)`.
- The original row gets `rn = 1` (retained); duplicates get `rn > 1` (deleted).

In [8]:
# Step 1: Audit duplicate review rows
q("""
SELECT listing_id, review_date, rating, review_text, COUNT(*) AS duplicate_count
FROM reviews
GROUP BY listing_id, review_date, rating, review_text
HAVING COUNT(*) > 1
LIMIT 5;
""")


,listing_id,review_date,rating,review_text,duplicate_count
0,4,02/23/2026,4,Exceeded my expectations.,2
1,29,2026.03.12,4,Good value for the price.,2
2,38,2026.03.07,5,"Works great, exactly as described.",2
3,39,05 March 2026,4,"Works great, exactly as described.",2
4,46,12 January 2026,2,"Arrived damaged, disappointed.",2


In [9]:
# Step 2: See ROW_NUMBER() in action on duplicates
q("""
SELECT review_id, listing_id, review_date, rating,
       ROW_NUMBER() OVER (
           PARTITION BY listing_id, review_date, rating, review_text 
           ORDER BY review_id
       ) AS row_num
FROM reviews
WHERE listing_id IN (
    SELECT listing_id FROM reviews
    GROUP BY listing_id, review_date, rating, review_text
    HAVING COUNT(*) > 1
)
LIMIT 6;
""")


,review_id,listing_id,review_date,rating,row_num
0,444,4,01/06/2026,4,1
1,291,4,01/29/2026,5,1
2,210,4,02/23/2026,4,1
3,415,4,02/23/2026,4,2
4,481,29,13 February 2026,5,1
5,397,29,2026.02.15,5,1


In [10]:
# Step 3: Delete rows where row_num > 1
before_count = q("SELECT COUNT(*) AS total FROM reviews").iloc[0]["total"]

conn.execute("""
    DELETE FROM reviews
    WHERE review_id IN (
        SELECT review_id FROM (
            SELECT review_id,
                   ROW_NUMBER() OVER (
                       PARTITION BY listing_id, review_date, rating, review_text
                       ORDER BY review_id
                   ) AS rn
            FROM reviews
        )
        WHERE rn > 1
    );
""")
conn.commit()

after_count = q("SELECT COUNT(*) AS total FROM reviews").iloc[0]["total"]
print(f"Reviews before: {before_count} | Reviews after: {after_count} | Deleted: {before_count - after_count}")


Reviews before: 595 | Reviews after: 574 | Deleted: 21


## Q6: How do we standardize 4 messy date formats into ISO `YYYY-MM-DD`?

### Concept Pointers:
- Dates arrive in 4 different formats (`YYYY-MM-DD`, `MM/DD/YYYY`, `YYYY.MM.DD`, `DD Month YYYY`).
- We parse all with Python `datetime.strptime()` and update SQLite with ISO standard strings.

In [11]:
# Inspect inconsistent date styles
q("SELECT DISTINCT review_date FROM reviews LIMIT 6")


,review_date
0,07 January 2026
1,03/15/2026
2,2026-02-22
3,2026.01.20
4,17 March 2026
5,14 January 2026


In [12]:
import datetime

def parse_to_iso(date_str):
    """Converts multiple date formats into ISO YYYY-MM-DD."""
    formats = ["%Y-%m-%d", "%m/%d/%Y", "%Y.%m.%d", "%d %B %Y"]
    for fmt in formats:
        try:
            return datetime.datetime.strptime(date_str, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None

# Test function on sample date strings
sample_dates = ["2026-03-01", "03/01/2026", "2026.03.01", "1 March 2026"]
[parse_to_iso(d) for d in sample_dates]


['2026-03-01', '2026-03-01', '2026-03-01', '2026-03-01']

In [13]:
# Apply to all reviews and update database
reviews_df = q("SELECT review_id, review_date FROM reviews")
reviews_df["iso_date"] = reviews_df["review_date"].apply(parse_to_iso)

for _, row in reviews_df.iterrows():
    conn.execute("UPDATE reviews SET review_date = ? WHERE review_id = ?",
                 (row["iso_date"], row["review_id"]))
conn.commit()

print("[STANDARDIZED] ISO dates verified in SQLite:")
q("SELECT review_id, review_date FROM reviews ORDER BY review_date LIMIT 5")


[STANDARDIZED] ISO dates verified in SQLite:


,review_id,review_date
0,111,2026-01-05
1,128,2026-01-05
2,163,2026-01-05
3,355,2026-01-05
4,409,2026-01-05


## Q7: How do we run a monthly review time-series aggregation query?

### Concept Pointers:
- With ISO dates, SQLite's `strftime('%Y-%m', review_date)` groups chronologically.

In [14]:
q("""
SELECT 
    strftime('%Y-%m', review_date) AS review_month,
    COUNT(*) AS total_reviews,
    ROUND(AVG(rating), 2) AS avg_monthly_rating
FROM reviews
GROUP BY review_month
ORDER BY review_month;
""")


,review_month,total_reviews,avg_monthly_rating
0,2026-01,211,3.82
1,2026-02,217,3.88
2,2026-03,146,3.76


## Summary & Key Takeaways

- **Normalized Schema**: Built 3-table normalized schema (`categories`, `products`, `reviews`).
- **Window Functions**: Ranked products within categories without losing row granularity.
- **CTEs**: Built multi-step Top Price Movers queries with clean readability.
- **In-Database Cleaning**: Safely deduplicated reviews with `ROW_NUMBER()` and standardized 4 date formats.
- **Hand-off to Sprint 4**: `ecommerce.db` is ready for Power BI Desktop!